## 1. Environment Setup

In [1]:
import sys 
import os
from pathlib import Path
import requests 
import time 
import json

print(f"Python Version: {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}")

current_dir = Path.cwd()
if current_dir.name == "week5" and current_dir.parent.name == "notebooks":
    project_root = current_dir.parent.parent
elif (current_dir / "compose.yml").exists():
    project_root = current_dir
else:
    project_root = Path("/Users/Shared/Projects/MOAI/zero_to_RAG")

if project_root.exists():
    print(f"Project root: {project_root}")
    sys.path.insert(0, str(project_root))
else:
    print("Project root not found - check directory structure")

print("✓ Environment setup complete")

# Default OpenRouter model slug for /ask requests from this notebook.
OPENROUTER_REQUEST_MODEL = os.environ.get("OPENROUTER_MODEL", "openai/gpt-4o-mini")


Python Version: 3.12.12
Project root: /Users/anhvietpham/Documents/AI/Project-practice/chatbot
✓ Environment setup complete


## 2. Service Health Check

In [2]:
# Check Service Health
print("WEEK 5 SERVICE HEALTH CHECK")
print("=" * 40)

services = {
    "FastAPI": "http://localhost:8001/api/v1/health",
    "OpenSearch": "http://localhost:9200/_cluster/health",
}

all_healthy = True
for service_name, url in services.items():
    try:
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            print(f"✓ {service_name}: Healthy")
        else:
            print(f"✗ {service_name}: HTTP {response.status_code}")
            all_healthy = False
    except:
        print(f"✗ {service_name}: Not accessible")
        all_healthy = False

if all_healthy:
    print("\n✓ All services ready for Week 5!")
else:
    print("\n⚠ Some services need attention. Run: docker compose up --build -d")

WEEK 5 SERVICE HEALTH CHECK
✓ FastAPI: Healthy
✓ OpenSearch: Healthy
✓ Ollama: Healthy

✓ All services ready for Week 5!


## 3. API Structure Overview

In [3]:
# Check API Endpoints
print("API STRUCTURE")
print("=" * 20)

try:
    response = requests.get("http://localhost:8001/openapi.json")
    if response.status_code == 200:
        openapi_data = response.json()
        endpoints = list(openapi_data['paths'].keys())
        
        print(f"Total endpoints: {len(endpoints)}")
        print("\nAvailable endpoints:")
        for endpoint in sorted(endpoints):
            print(f"  • {endpoint}")
    else:
        print(f"Could not fetch API info: {response.status_code}")
except Exception as e:
    print(f"Error: {e}")

API STRUCTURE
Total endpoints: 4

Available endpoints:
  • /api/v1/ask
  • /api/v1/health
  • /api/v1/hybrid-search/
  • /api/v1/stream


## 4. Test LLM (OpenRouter via API)

In [4]:
print("OPENROUTER / LLM CHECK (via API /health)")
print("=" * 20)

try:
    h = requests.get("http://localhost:8001/api/v1/health", timeout=10).json()
    llm = (h.get("services") or {}).get("llm")
    if llm:
        print(f"LLM: {llm.get('status')} — {llm.get('message')}")
    else:
        print("No llm entry in health response")
except Exception as e:
    print(f"Error: {e}")


OLLAMA LLM TEST
Available models: [{'name': 'llama3.2:1b', 'model': 'llama3.2:1b', 'modified_at': '2026-03-20T10:10:50.165010013Z', 'size': 1321098329, 'digest': 'baf6a787fdffd633537aa2eb51cfd54cb93ff08e28040095462bb63daf552878', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'llama', 'families': ['llama'], 'parameter_size': '1.2B', 'quantization_level': 'Q8_0'}}]
 * llama3.2:1b


In [9]:
print("\nTesting LLM via POST /api/v1/ask:")

try:
    response = requests.post(
        "http://localhost:8001/api/v1/ask",
        json={
            "query": "What is 2+6? Reply with one digit only.",
            "top_k": 1,
            "use_hybrid": True,
            "model": OPENROUTER_REQUEST_MODEL,
        },
        timeout=120,
    )
    if response.status_code == 200:
        answer = (response.json().get("answer") or "").strip()
        print(f"✓ LLM answered: {answer[:200]}")
        print("✓ OpenRouter path is working (requires OPENROUTER_API_KEY on API).")
    else:
        print(f"✗ /ask failed: {response.status_code} — {response.text[:500]}")
except Exception as e:
    print(f"✗ Error: {e}")


Testing LLM Generation:
✓ LLM responded: '8'
✓ Ollama is working!


## 5. Test Search Functionally

In [10]:
print("SEARCH TEST")
print("=" * 15)

search_query = "machine learning"
print(f"Searching for: '{search_query}'")

try: 
    search_request = {
        "query": search_query, 
        "use_hybrid": True, 
        "size": 3
    }

    response = requests.post(
        "http://localhost:8001/api/v1/hybrid-search/",
        json=search_request,
        timeout=30
    )  

    if response.status_code == 200: 
        data = response.json()
        print(f" Found {data['total']} results")
        print(f" Search mode: {data['search_mode']}")

        if data['hits']:
            print("\nTop results:")
            for i, hit in enumerate(data['hits'][:2], 1):
                title = hit.get('title', 'Unknown')[:60]
                score = hit.get('score', 0)
                print(f"  {i}. {title}... (score: {score:.3f})")
        else:
            print("No results found")
    else:
        print(f"✗ Search failed: {response.status_code}")
        
except Exception as e:
    print(f"✗ Error: {e}")



SEARCH TEST
Searching for: 'machine learning'
 Found 3 results
 Search mode: hybrid

Top results:
  1. Hierarchical Cooperative Multi-Agent Reinforcement Learning ... (score: 0.032)
  2. Hierarchical Cooperative Multi-Agent Reinforcement Learning ... (score: 0.032)


## 6. Complete RAG Pipeline Test

In [11]:
print("COMPLETE RAG PIPELINE TEST (OPTIMIZED)")
print("=" * 30)

question = "Summarize machine learning research papers?"
print(f"Question: '{question}'")

start_time = time.time()

try: 
    rag_request = {
        "query": question,
        "top_k": 1,  # Use 1 chunk for context
        "use_hybrid": True,  # Use best search
        "model": OPENROUTER_REQUEST_MODEL
    }

    response = requests.post(
        "http://localhost:8001/api/v1/ask/",
        json=rag_request,
        timeout=60
    )

    response_time = time.time() - start_time

    if response.status_code == 200:
        data = response.json()
        
        print(f"\n✓ Success! ({response_time:.1f} seconds)")
        print(f"\nAnswer:")
        print("-" * 40)
        print(data['answer'])
        print("-" * 40)
        
        print(f"\nSources: {len(data.get('sources', []))} papers")
        print(f"Chunks used: {data.get('chunks_used', 0)}")
        print(f"Search mode: {data.get('search_mode', 'unknown')}")

    else:
        print(f"\n✗ Request failed: HTTP {response.status_code}")
        print(f"Response: {response.text[:200]}")
        
except Exception as e:
    print(f"\n✗ Error: {e}")

COMPLETE RAG PIPELINE TEST (OPTIMIZED)
Question: 'Summarize machine learning research papers?'

✓ Success! (44.4 seconds)

Answer:
----------------------------------------
Machine learning has been a rapidly advancing field in recent years, with numerous papers exploring its applications and advancements. Papers like [40] by Tang et al. discuss hierarchical deep multiagent reinforcement learning, which leverages hierarchical structures to improve agent performance. On the other hand, [41] introduces feudal networks for hierarchical reinforcement learning, where agents learn through interacting with each other's knowledge.

Other notable papers include [42] by Watkins and Dayan on Q-learning, [43] by Wilson et al. on Bayesian policy search, and [45] by Zhang et al. on integrating independent and centralized multi-agent reinforcement learning methods. These works demonstrate the versatility of machine learning in solving complex problems across various domains.

Additionally, papers like

## 7. Complete RAG Pipeline Test - Streaming

In [12]:
print("COMPLETE RAG PIPELINE TEST (STREAMING)")
print("=" * 30)

question = "Summarize machine learning papers?"
print(f"Question: '{question}'")

start_time = time.time()
try: 
    rag_request = {
        "query": question,
        "top_k": 1,  # Use 1 chunk for context
        "use_hybrid": True,  # Use best search
        "model": OPENROUTER_REQUEST_MODEL
    }

    response = requests.post(
        "http://localhost:8001/api/v1/stream",
        json=rag_request,
        stream=True,  # Enable streaming
        timeout=120
    )

    if response.status_code == 200: 
        full_answer = ""
        sources = []
        chunks_used = 0
        search_mode = "unknown"
        first_chunk_time = None

        print(f"\nStreaming response...")

        for line in response.iter_lines(): 
            if line: 
                line_str = line.decode('utf-8')
                if line_str.startswith("data: "): 
                    try: 
                        data = json.loads(line_str[6:])

                        # Handle metadata
                        if 'sources' in data: 
                            sources = data['sources']
                            chunks_used = data.get('chunks_used', 0)
                            search_mode = data.get('search_mode', 'unknown')

                        if 'chunk' in data: 
                            if first_chunk_time is None: 
                                first_chunk_time = time.time() - start_time
                                print(f"First response in: {first_chunk_time:.1f} seconds")
                                print("\nAnswer:")
                                print("-" * 40)
                            
                            chunk_text = data['chunk']
                            full_answer += chunk_text
                            print(chunk_text, end='', flush=True)
                        
                        if data.get('done', False): 
                            break
                            
                    except json.JSONDecodeError:
                        continue

        response_time = time.time() - start_time

        print("\n" + "-" * 40)
        print(f"\n✓ Complete! (Total: {response_time:.1f} seconds)")
        
        print(f"\nSources: {len(sources)} papers")
        if sources:
            for i, source in enumerate(sources[:2], 1):
                print(f"  {i}. {source}")
        print(f"Chunks used: {chunks_used}")
        print(f"Search mode: {search_mode}")

    else:
        print(f"\n✗ Request failed: HTTP {response.status_code}")
        print(f"Response: {response.text[:200]}")
        
except Exception as e:
    print(f"\n✗ Error: {e}")
    import traceback
    traceback.print_exc()

COMPLETE RAG PIPELINE TEST (STREAMING)
Question: 'Summarize machine learning papers?'

Streaming response...
First response in: 14.2 seconds

Answer:
----------------------------------------
Machine learning has made tremendous progress in recent years, with various techniques being applied to different domains. Researchers have explored approaches such as deep reinforcement learning, hierarchical multi-agent learning, and value function factorization. These methods have shown promising results in complex environments like robotics, control systems, and games.

One notable example is the work on hierarchical multi-agent reinforcement learning by Makar et al. [22] who proposed a framework for hierarchical learning in cooperative multi-agent systems. They used a hierarchical representation of knowledge to improve decision-making and coordination among agents.

Another area of research involves value function factorization, as seen in papers by Stolle and Precup [34], which introduced the

In [13]:
print("SYSTEM STATUS SUMMARY")
print("=" * 20)

try: 
    health_response = requests.get("http://localhost:8001/api/v1/health")
    if health_response.status_code == 200: 
        health_data = health_response.json()

        print(f"Overall Status: {health_data.get('status', 'unknown').upper()}")
        print(f"Version: {health_data.get('version', 'unknown')}")

        print("\nServices status:")
        services = health_data.get('services', {})
        for service, info in services.items(): 
            status = info.get('status', 'unknown')
            message = info.get('message', '')
            print(f" * {service}: {status} - {message}")

        print("\nRAG Pipeline Status:")
        print("  ✓ Data Ingestion: Papers indexed in OpenSearch")
        print("  ✓ Search: BM25 + Vector hybrid search working")
        print("  ✓ LLM Generation: OpenRouter (when OPENROUTER_API_KEY is set on API)")
        print("  ✓ Performance: 6x speed improvement (120s → 15-20s)")
        print("  ✓ API: Clean endpoints ready for production")

        # Check endpoint availability
        print("\nEndpoint Status:")
        try:
            test_response = requests.get("http://localhost:8001/openapi.json")
            if test_response.status_code == 200:
                endpoints = list(test_response.json()['paths'].keys())
                print(f"  ✓ Standard RAG: /api/v1/ask/ (working)")
                
                if "/api/v1/ask/ask-stream/" in endpoints:
                    print(f"  ✓ Streaming RAG: /api/v1/ask/ask-stream/ (available)")
                else:
                    print(f"  ⚠ Streaming RAG: /api/v1/ask/ask-stream/ (needs container rebuild)")
                
                print(f"  ✓ Search: /api/v1/hybrid-search/ (working)")
        except:
            print("  ⚠ Could not check endpoint status")
        
        print("\n🎉 Complete RAG system operational!")
        print(f"   • Dramatic performance improvement achieved")
        print(f"   • Production-ready with excellent response times")
        
    else:
        print(f"Could not get system status: {health_response.status_code}")
        
except Exception as e:
    print(f"Error checking system status: {e}")


SYSTEM STATUS SUMMARY
Overall Status: OK
Version: 0.1.0

Services status:
 * database: healthy - Connected successfully
 * opensearch: healthy - Index 'chatbot-papers-chunks_v2' with 18 documents
 * ollama: healthy - Ollama service is running

RAG Pipeline Status:
  ✓ Data Ingestion: Papers indexed in OpenSearch
  ✓ Search: BM25 + Vector hybrid search working
  ✓ LLM Generation: Ollama generating answers
  ✓ Performance: 6x speed improvement (120s → 15-20s)
  ✓ API: Clean endpoints ready for production

Endpoint Status:
  ⚠ Could not check endpoint status

🎉 Complete RAG system operational!
   • Dramatic performance improvement achieved
   • Production-ready with excellent response times


## 8. Using the Gradio Interface

In [14]:
# Launch Gradio Interface Instructions

print("GRADIO INTERFACE")
print("=" * 40)

print("\n📱 Web Interface Available!")
print("\nTo use the Gradio interface:")
print("1. Open a terminal")
print("2. Run: uv run python gradio_launcher.py")
print("3. Open browser to: http://localhost:7861")
print("\nFeatures:")
print("  • Real-time streaming responses")
print("  • Interactive parameter controls")
print("  • Clean, user-friendly design")
print("  • Example questions included")
print("  • Source paper links")

# Check if Gradio is running
try:
    gradio_check = requests.get("http://localhost:7861", timeout=2)
    if gradio_check.status_code == 200:
        print("\n✅ Gradio interface is running!")
        print("   Visit: http://localhost:7861")
    else:
        print("\n⚠️ Gradio not detected on port 7861")
        print("   Run: uv run python gradio_launcher.py")
except:
    print("\n⚠️ Gradio interface not running")
    print("   To start: uv run python gradio_launcher.py")
    

GRADIO INTERFACE

📱 Web Interface Available!

To use the Gradio interface:
1. Open a terminal
2. Run: uv run python gradio_launcher.py
3. Open browser to: http://localhost:7861

Features:
  • Real-time streaming responses
  • Interactive parameter controls
  • Clean, user-friendly design
  • Example questions included
  • Source paper links

⚠️ Gradio interface not running
   To start: uv run python gradio_launcher.py
